# Comparación A · B · C · D — Resultados de la campaña experimental

Notebook de análisis y figuras para la tesis. Lee los CSV producidos por:
- `scripts/run_campaign.py` → `outputs/tables/<campaign>.csv` (una fila por run).
- `scripts/aggregate_results.py` → `outputs/tables/<campaign>_summary.csv` y `<campaign>_paired_c_vs_d.csv`.

Las figuras se exportan a `outputs/figures/` para incluirlas en el documento (Cap. 8.24).

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from src.utils.paths import FIGURES_DIR, TABLES_DIR, ensure_dir

ensure_dir(FIGURES_DIR)

# Configurable: cambia CAMPAIGN_NAME para apuntar a otra corrida
CAMPAIGN_NAME = "mini"
runs_csv = TABLES_DIR / f"{CAMPAIGN_NAME}.csv"
summary_csv = TABLES_DIR / f"{CAMPAIGN_NAME}_summary.csv"
paired_csv = TABLES_DIR / f"{CAMPAIGN_NAME}_paired_c_vs_d.csv"

runs = pd.read_csv(runs_csv)
summary = pd.read_csv(summary_csv)
paired = pd.read_csv(paired_csv) if paired_csv.is_file() else pd.DataFrame()
print(f"Runs: {len(runs)}  |  Modelos: {sorted(runs['model'].unique())}")
runs.head()

## 1. Métricas financieras por modelo

In [ ]:
fin_metrics = ["cumulative_return", "sharpe_ratio", "max_drawdown"]
fig, axes = plt.subplots(1, len(fin_metrics), figsize=(12, 4))
for ax, metric in zip(axes, fin_metrics):
    data = [runs.loc[runs["model"] == m, metric].values for m in ("A", "B", "C", "D")]
    ax.boxplot(data, tick_labels=["A", "B", "C", "D"])
    ax.set_title(metric)
    ax.grid(alpha=0.3)
fig.tight_layout()
fig.savefig(FIGURES_DIR / f"{CAMPAIGN_NAME}_financial.png", dpi=120)
plt.show()

## 2. Métricas de exploración (top-m hit-rate, candidate hit-rate, cobertura)

In [ ]:
exp_metrics = ["topm_hit_rate", "candidate_hit_rate", "asset_coverage"]
fig, axes = plt.subplots(1, len(exp_metrics), figsize=(12, 4))
for ax, metric in zip(axes, exp_metrics):
    data = [runs.loc[runs["model"] == m, metric].values for m in ("A", "B", "C", "D")]
    ax.boxplot(data, tick_labels=["A", "B", "C", "D"])
    ax.set_title(metric)
    ax.set_ylim(0.0, 1.05)
    ax.grid(alpha=0.3)
fig.tight_layout()
fig.savefig(FIGURES_DIR / f"{CAMPAIGN_NAME}_exploration.png", dpi=120)
plt.show()

## 3. Latencia por step (módulo local)

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
data = [runs.loc[runs["model"] == m, "mean_latency_ms"].values for m in ("A", "B", "C", "D")]
ax.boxplot(data, tick_labels=["A", "B", "C", "D"])
ax.set_title("Latencia por step (ms)")
ax.set_ylabel("ms")
ax.grid(alpha=0.3)
fig.tight_layout()
fig.savefig(FIGURES_DIR / f"{CAMPAIGN_NAME}_latency.png", dpi=120)
plt.show()

## 4. Comparación pareada D vs C (Sec. 8.18)

Diferencia D − C por semilla con IC95% por bootstrap pareado (5000 iter).

In [ ]:
if paired.empty:
    print("No hay paired_c_vs_d.csv para esta campaña.")
else:
    paired_view = paired.set_index("metric")[[
        "mean_diff_d_minus_c", "ci_lo", "ci_hi", "n_pairs", "p_d_better_than_c"
    ]]
    display(paired_view)

## 5. Resumen ejecutivo

(Pendiente: rellenar con interpretación de resultados según Sec. 8.22 — criterios de interpretación.)